# Demo Person Re-Identification

## 1. Mục tiêu notebook cho video/thuyết trình

Notebook này được thiết kế để **kể chuyện đồ án** theo flow thuyết trình, không chỉ chạy lệnh thực nghiệm.

Flow chính:

1. Bài toán Person Re-ID và khó khăn thực tế.
2. Baseline TransReID và 3 hướng cải tiến.
3. Kiểm tra dataset/checkpoint trước khi demo.
4. Chạy evaluation từ checkpoint có sẵn.
5. Phân tích metric + visualization định lượng/định tính.
6. Kết luận, hạn chế và script gợi ý khi quay video.

Lưu ý: phần train được chuyển thành **tham khảo tùy chọn** ở gần cuối notebook, không phải trọng tâm khi quay video.

## 2. Bài toán Person Re-Identification là gì?

Trong Person Re-ID, ta có:

- **Query**: ảnh người cần truy hồi.
- **Gallery**: tập ảnh ứng viên từ nhiều camera.
- **Embedding**: vector đặc trưng của ảnh.
- **Ranking**: sắp xếp gallery theo độ tương đồng với query.

Điểm quan trọng: Re-ID **khác** classification. Mục tiêu không phải dự đoán 1 nhãn duy nhất, mà là trả về **danh sách gallery được xếp hạng** theo mức độ giống query.

Các khó khăn chính trong Re-ID:

- Cross-camera domain shift (khác góc nhìn/cảm biến).
- Pose variation (tư thế khác nhau).
- Illumination change (ánh sáng thay đổi).
- Occlusion (bị che khuất một phần).
- Background clutter (nền phức tạp).
- Visual ambiguity (người khác mặc đồ tương tự).

## 3. Cấu hình điều khiển demo

Các biến `RUN_*` quyết định notebook có thực sự chạy lệnh nặng hay chỉ in lệnh và hiển thị artifact đã có.

- `RUN_EVAL = True`: chạy `test.py` cho các checkpoint.
- `RUN_VISUALIZE = True`: chạy lại các script tạo ảnh visualization.
- `RUN_TRAIN = True`: chạy train thử. Mặc định tắt vì rất tốn thời gian.
- `FAST_TRAIN_DEMO = True`: nếu bật train, chỉ chạy 1 epoch để kiểm tra pipeline, không dùng làm kết quả chính thức.

In [ ]:
from pathlib import Path
import os
import sys
import csv
import json
import shlex
import random
import subprocess
from collections import defaultdict
from datetime import datetime

from IPython.display import display, Markdown, Image as IPyImage

PROJECT_ROOT = Path(r"D:\DAI_HOC\NAM_3\SEM 2\TGMT\PROJECT\K23-TGMT")
SOURCE_ROOT = PROJECT_ROOT / "1.3 Source code"
DEMO_ROOT = PROJECT_ROOT / "1.4 Demo"
ARTIFACT_ROOT = SOURCE_ROOT / "transreid_artifacts"

LOCAL_SRC = SOURCE_ROOT / "local-reliability"
SEMANTIC_SRC = SOURCE_ROOT / "semantic"
TOOLS_DIR = ARTIFACT_ROOT / "tools"

DATASET_ROOT = Path(r"D:\DAI_HOC\NAM_3\SEM 2\TGMT\PROJECT\Requirement 2 & 3\Colab\data\market1501")
DATA_PARENT = DATASET_ROOT.parent

DEMO_OUTPUT = DEMO_ROOT / "generated_outputs"
DEMO_OUTPUT.mkdir(parents=True, exist_ok=True)

RUN_INSTALL = False
RUN_TRAIN = False
FAST_TRAIN_DEMO = True
RUN_EVAL = False
RUN_VISUALIZE = False

DEVICE_ID = "0"
PYTHON = sys.executable

print("Project:", PROJECT_ROOT)
print("Dataset exists:", DATASET_ROOT.exists(), DATASET_ROOT)
print("Artifacts:", ARTIFACT_ROOT)

## 4. Baseline và 3 nhánh đề xuất

Ba hướng mở rộng được demo là:

1. **Local Reliability**: điều chỉnh đóng góp của patch cục bộ theo độ tin cậy.
2. **Semantic Alignment**: gán semantic body-part cho patch token.
3. **Semantic + Reliability**: kết hợp hai tín hiệu trên.

Baseline TransReID được giữ để so sánh trực tiếp.

In [ ]:
BRANCHES = {
    "baseline": {
        "title": "Baseline TransReID",
        "source": SEMANTIC_SRC,
        "config": ARTIFACT_ROOT / "configs_snapshot" / "Market" / "vit_transreid_stride_fair.yml",
        "checkpoint": ARTIFACT_ROOT / "runs" / "fair_market_baseline" / "transformer_best.pth",
        "output": DEMO_OUTPUT / "eval_baseline",
        "note": "Mô hình nền TransReID.",
    },
    "local": {
        "title": "Local Reliability",
        "source": LOCAL_SRC,
        "config": ARTIFACT_ROOT / "configs_snapshot" / "Market" / "vit_transreid_stride_local_reliability.yml",
        "checkpoint": ARTIFACT_ROOT / "runs" / "fair_market_local_reliability" / "transformer_best.pth",
        "output": DEMO_OUTPUT / "eval_local_reliability",
        "note": "Ước lượng reliability score cho patch/local token.",
    },
    "semantic": {
        "title": "Semantic Alignment",
        "source": SEMANTIC_SRC,
        "config": ARTIFACT_ROOT / "configs_snapshot" / "Market" / "vit_transreid_stride_sem_align.yml",
        "checkpoint": ARTIFACT_ROOT / "runs" / "fair_market_sem_align" / "transformer_best.pth",
        "output": DEMO_OUTPUT / "eval_semantic_alignment",
        "note": "Dùng semantic mask để giám sát patch token.",
    },
    "combine": {
        "title": "Semantic + Reliability",
        "source": SEMANTIC_SRC,
        "config": ARTIFACT_ROOT / "configs_snapshot" / "Market" / "vit_transreid_stride_sem_align_reliability.yml",
        "checkpoint": ARTIFACT_ROOT / "runs" / "fair_market_sem_align_reliability" / "transformer_best.pth",
        "output": DEMO_OUTPUT / "eval_semantic_reliability",
        "note": "Kết hợp semantic supervision và reliability-aware local fusion.",
    },
}

rows = []
for key, branch in BRANCHES.items():
    ckpt = branch["checkpoint"]
    cfg = branch["config"]
    rows.append({
        "key": key,
        "model": branch["title"],
        "source": str(branch["source"]),
        "config_exists": cfg.exists(),
        "checkpoint_exists": ckpt.exists(),
        "checkpoint_size_MB": round(ckpt.stat().st_size / 1024**2, 2) if ckpt.exists() else None,
        "note": branch["note"],
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    for row in rows:
        print(row)

## 5. Preflight check trước khi quay demo

Cell này kiểm tra nhanh toàn bộ thành phần cần thiết:

- Đường dẫn project/source/artifact/tool.
- Cấu trúc dataset và số ảnh từng split.
- Config + checkpoint cho 4 nhánh.
- Các script visualization quan trọng.

Ký hiệu trạng thái:

- `[OK]`: đầy đủ, sẵn sàng.
- `[WARNING]`: thiếu thành phần phụ, vẫn có thể demo phần chính.
- `[MISSING]`: thiếu thành phần bắt buộc cho demo checkpoint, sẽ báo lỗi rõ ràng.

In [ ]:
def count_images(split_name):
    split_dir = DATASET_ROOT / split_name
    if not split_dir.exists():
        return 0
    return len(list(split_dir.glob("*.jpg")))


def preflight_check(verbose=True, strict=True):
    logs = []
    hard_missing = []

    def push(status, label, path=None, required=False, detail=""):
        msg = f"[{status}] {label}"
        if path is not None:
            msg += f" -> {path}"
        if detail:
            msg += f" ({detail})"
        logs.append(msg)
        if status == "MISSING" and required:
            hard_missing.append(msg)

    # Core paths
    core_paths = [
        ("PROJECT_ROOT", PROJECT_ROOT, True),
        ("SOURCE_ROOT", SOURCE_ROOT, True),
        ("ARTIFACT_ROOT", ARTIFACT_ROOT, True),
        ("LOCAL_SRC", LOCAL_SRC, True),
        ("SEMANTIC_SRC", SEMANTIC_SRC, True),
        ("TOOLS_DIR", TOOLS_DIR, False),
    ]
    for name, p, required in core_paths:
        push("OK" if p.exists() else "MISSING", name, p, required=required)

    # Dataset and splits
    push("OK" if DATASET_ROOT.exists() else "MISSING", "DATASET_ROOT", DATASET_ROOT, required=True)
    for split in ["bounding_box_train", "query", "bounding_box_test"]:
        split_dir = DATASET_ROOT / split
        if split_dir.exists():
            push("OK", f"Split {split}", split_dir, detail=f"{count_images(split)} images")
        else:
            required = split in {"query", "bounding_box_test"}
            push("MISSING", f"Split {split}", split_dir, required=required)

    # Branch assets
    for key, branch in BRANCHES.items():
        cfg = branch["config"]
        ckpt = branch["checkpoint"]
        push("OK" if cfg.exists() else "MISSING", f"{key}.config", cfg, required=True)
        if ckpt.exists():
            mb = round(ckpt.stat().st_size / 1024**2, 2)
            push("OK", f"{key}.checkpoint", ckpt, required=True, detail=f"{mb} MB")
        else:
            push("MISSING", f"{key}.checkpoint", ckpt, required=True)

    # Important scripts for visualization
    script_checks = [
        ("eval.local.test", LOCAL_SRC / "test.py", True),
        ("eval.semantic.test", SEMANTIC_SRC / "test.py", True),
        ("viz.report", TOOLS_DIR / "generate_report_visuals.py", False),
        ("viz.multimodel_topk", TOOLS_DIR / "visualize_multimodel_retrieval.py", False),
        ("viz.local_suite", LOCAL_SRC / "visualize_person_reid_suite.py", False),
        ("viz.semantic_basic", SEMANTIC_SRC / "visualize_semantic_no_checkpoint.py", False),
        ("viz.semantic_real", SEMANTIC_SRC / "visualize_real_semantic_examples.py", False),
        ("viz.semantic_extra", SEMANTIC_SRC / "visualize_semantic_extra.py", False),
    ]
    for label, p, required in script_checks:
        status = "OK" if p.exists() else ("MISSING" if required else "WARNING")
        push(status, label, p, required=required)

    if verbose:
        print("Python:", sys.version.split()[0])
        try:
            import torch
            print("Torch:", torch.__version__)
            print("CUDA available:", torch.cuda.is_available())
            if torch.cuda.is_available():
                print("GPU:", torch.cuda.get_device_name(0))
        except Exception as exc:
            print("[WARNING] Torch check failed:", exc)

        print("\n--- Preflight Summary ---")
        for line in logs:
            print(line)

    if hard_missing and strict:
        raise RuntimeError(
            "Thiếu thành phần bắt buộc cho demo checkpoint. Hãy sửa các mục [MISSING] sau:\n"
            + "\n".join(hard_missing)
        )

    return {"logs": logs, "hard_missing": hard_missing}


_ = preflight_check(verbose=True, strict=True)

### Helper functions (internal)

Các hàm tiện ích bên dưới dùng để chạy lệnh và hiển thị ảnh/CSV gọn trong notebook. Phần này hỗ trợ kỹ thuật, không phải nội dung thuyết trình chính.

In [ ]:
def quote_cmd(cmd):
    return " ".join(shlex.quote(str(x)) for x in cmd)

def run_command(cmd, cwd, run=False, env=None):
    print("CWD:", cwd)
    print("CMD:", quote_cmd(cmd))
    if not run:
        print("Skip: đặt cờ RUN_* = True để chạy lệnh này.")
        return None
    env_all = os.environ.copy()
    if env:
        env_all.update(env)
    proc = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        env=env_all,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(proc.stdout[-6000:])
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed with return code {proc.returncode}")
    return proc

def show_image(path, width=1000):
    path = Path(path)
    if not path.exists():
        display(Markdown(f"**Chưa có ảnh:** `{path}`"))
        return
    display(Markdown(f"`{path}`"))
    display(IPyImage(filename=str(path), width=width))

def show_csv(path):
    path = Path(path)
    if not path.exists():
        print("Missing:", path)
        return
    try:
        import pandas as pd
        display(pd.read_csv(path))
    except Exception:
        print(path.read_text(encoding="utf-8-sig"))

## 6. Dataset preview + same identity across cameras

Mục tiêu phần này là minh họa trực quan bản chất Re-ID:

- Cùng một `pid` có thể nhìn rất khác khi qua camera khác nhau.
- Đồng thời có nhiều `pid` khác trông khá giống nhau (distractors), gây khó cho retrieval.

Cell bên dưới sẽ:

1. Parse `pid` và `camid` từ filename kiểu Market-1501/Duke.
2. Chọn một identity có nhiều ảnh, ưu tiên nhiều camera.
3. Hiển thị 4-6 ảnh cùng `pid` ở nhiều camera.
4. Hiển thị thêm một hàng ảnh khác `pid` để minh họa visual ambiguity.

In [ ]:
def parse_pid_camid(img_path):
    stem = Path(img_path).stem
    parts = stem.split("_")
    pid = parts[0] if parts else "unknown"
    camid = "unknown"
    for part in parts[1:]:
        if part.startswith("c") and len(part) > 1 and part[1].isdigit():
            camid = part[1:]
            break
    return pid, camid


def list_split_images(split_name, limit=None):
    split_dir = DATASET_ROOT / split_name
    if not split_dir.exists():
        return []
    images = sorted(split_dir.glob("*.jpg"))
    if limit is not None:
        return images[:limit]
    return images


def build_pid_index(images):
    index = defaultdict(list)
    for p in images:
        pid, camid = parse_pid_camid(p)
        if pid in {"-1", "0000", "unknown"}:
            continue
        index[pid].append((p, camid))
    return index


def show_same_id_cross_camera(max_same_id=6, max_distractor=6):
    try:
        import matplotlib.pyplot as plt
        from PIL import Image
    except Exception as exc:
        print("[WARNING] Missing matplotlib/Pillow for preview:", exc)
        return

    gallery_images = list_split_images("bounding_box_test")
    if not gallery_images:
        print("[MISSING] Không tìm thấy ảnh trong split bounding_box_test.")
        return

    pid_index = build_pid_index(gallery_images)
    candidates = []
    for pid, items in pid_index.items():
        cams = {cam for _, cam in items}
        if len(items) >= 4:
            candidates.append((pid, len(cams), len(items), items))

    if not candidates:
        print("[WARNING] Không đủ identity có >=4 ảnh để minh họa cross-camera.")
        return

    # Ưu tiên identity có nhiều camera nhất, sau đó nhiều ảnh nhất.
    candidates.sort(key=lambda x: (x[1], x[2]), reverse=True)
    chosen_pid, num_cams, _, items = candidates[0]

    same_samples = sorted(items, key=lambda x: x[1])[:max_same_id]

    other_pids = [pid for pid in pid_index.keys() if pid != chosen_pid]
    distractors = []
    for pid in other_pids:
        if len(distractors) >= max_distractor:
            break
        distractors.append((pid_index[pid][0][0], parse_pid_camid(pid_index[pid][0][0])[1], pid))

    n_cols = max(len(same_samples), len(distractors), 4)
    fig, axes = plt.subplots(2, n_cols, figsize=(3.2 * n_cols, 8))

    for c in range(n_cols):
        axes[0, c].axis("off")
        axes[1, c].axis("off")

    for c, (img_path, camid) in enumerate(same_samples):
        axes[0, c].imshow(Image.open(img_path).convert("RGB"))
        axes[0, c].set_title(f"same pid={chosen_pid}\ncam={camid}", fontsize=10)

    for c, (img_path, camid, pid) in enumerate(distractors):
        axes[1, c].imshow(Image.open(img_path).convert("RGB"))
        axes[1, c].set_title(f"distractor pid={pid}\ncam={camid}", fontsize=10)

    fig.suptitle(
        f"Same identity across cameras (pid={chosen_pid}, cams={num_cams}) vs distractors",
        fontsize=14,
    )
    plt.tight_layout()
    plt.show()


show_same_id_cross_camera(max_same_id=6, max_distractor=6)

## 7. Query-gallery setup

Trước khi xem kết quả model, cần nhìn rõ setup truy hồi:

- **Query image**: ảnh mục tiêu cần tìm.
- **Gallery candidates**: tập ảnh đối chiếu.
- Model sẽ xếp hạng gallery theo độ tương đồng embedding.

Cell dưới đây hiển thị một query và vài gallery candidates (gồm đúng/sai identity) để minh họa trực quan bài toán ranking.

In [ ]:
def show_query_gallery_example(num_gallery=5):
    try:
        import matplotlib.pyplot as plt
        from PIL import Image
    except Exception as exc:
        print("[WARNING] Missing matplotlib/Pillow for query-gallery preview:", exc)
        return

    query_images = list_split_images("query")
    gallery_images = list_split_images("bounding_box_test")

    if not query_images or not gallery_images:
        print("[MISSING] Cần có cả split query và bounding_box_test để minh họa query-gallery.")
        return

    gallery_index = build_pid_index(gallery_images)

    selected_query = None
    selected_positive = None
    for q in query_images:
        q_pid, q_cam = parse_pid_camid(q)
        if q_pid in {"-1", "0000", "unknown"}:
            continue
        positives = [item for item in gallery_index.get(q_pid, []) if item[0].name != q.name]
        if positives:
            selected_query = q
            selected_positive = positives[0][0]
            break

    if selected_query is None:
        print("[WARNING] Không tìm thấy query có positive match trong gallery.")
        return

    q_pid, q_cam = parse_pid_camid(selected_query)

    negatives = []
    for g in gallery_images:
        g_pid, g_cam = parse_pid_camid(g)
        if g_pid != q_pid and g_pid not in {"-1", "0000", "unknown"}:
            negatives.append(g)
        if len(negatives) >= max(1, num_gallery - 1):
            break

    gallery_show = [selected_positive] + negatives

    n_cols = 1 + len(gallery_show)
    fig, axes = plt.subplots(1, n_cols, figsize=(4.0 * n_cols, 5.2))

    axes[0].imshow(Image.open(selected_query).convert("RGB"))
    axes[0].set_title(f"QUERY\npid={q_pid} cam={q_cam}", fontsize=11)
    axes[0].axis("off")

    for i, g in enumerate(gallery_show, start=1):
        g_pid, g_cam = parse_pid_camid(g)
        is_match = g_pid == q_pid
        axes[i].imshow(Image.open(g).convert("RGB"))
        axes[i].set_title(
            f"GALLERY-{i}\npid={g_pid} cam={g_cam}\n{'MATCH' if is_match else 'DISTRACTOR'}",
            fontsize=10,
        )
        axes[i].axis("off")

    fig.suptitle("Query-Gallery setup for Person Re-ID", fontsize=14)
    plt.tight_layout()
    plt.show()


show_query_gallery_example(num_gallery=5)

## 8. Evaluation từ checkpoint đã huấn luyện

Đây là bước chính khi demo video. Notebook dùng checkpoint đã huấn luyện và gọi `test.py` để tính mAP, Rank-1, Rank-5, Rank-10.

Bật `RUN_EVAL = True` để chạy lại đánh giá. Kết quả sẽ được ghi vào thư mục `1.4 Demo/generated_outputs/eval_*`.

In [ ]:
def build_eval_command(key):
    branch = BRANCHES[key]
    cmd = [
        PYTHON, "test.py",
        "--config_file", branch["config"],
        "DATASETS.ROOT_DIR", str(DATA_PARENT),
        "MODEL.DEVICE_ID", DEVICE_ID,
        "TEST.WEIGHT", str(branch["checkpoint"]),
        "TEST.IMS_PER_BATCH", "128",
        "DATALOADER.NUM_WORKERS", "0",
        "OUTPUT_DIR", str(branch["output"]),
    ]
    return cmd

for key in ["baseline", "local", "semantic", "combine"]:
    display(Markdown(f"### Evaluate: {BRANCHES[key]['title']}"))
    cmd = build_eval_command(key)
    run_command(cmd, cwd=BRANCHES[key]["source"], run=RUN_EVAL)

## 9. Metric summary (mAP, Rank-1, Rank-5, Rank-10)

Ý nghĩa nhanh các metric chính:

- **mAP**: trung bình average precision trên nhiều query, phản ánh chất lượng ranking tổng thể.
- **Rank-1**: tỷ lệ query có ảnh đúng ở vị trí top-1.
- **Rank-5 / Rank-10**: tỷ lệ có ảnh đúng trong top-5 / top-10.

Với Re-ID, mAP thường thể hiện rõ hơn mức cải thiện toàn bộ danh sách truy hồi, thay vì chỉ top-1.

In [ ]:
show_csv(ARTIFACT_ROOT / "metrics_summary.csv")
show_csv(ARTIFACT_ROOT / "latency_summary.csv")

## 10. Visualization định lượng

Script `generate_report_visuals.py` đọc log/CSV đã có và sinh biểu đồ:

- mAP và CMC Rank-k.
- Delta so với baseline.
- Training/validation curves.
- Accuracy-latency trade-off.

Bật `RUN_VISUALIZE = True` để chạy lại.

In [ ]:
cmd = [PYTHON, str(TOOLS_DIR / "generate_report_visuals.py")]
run_command(cmd, cwd=ARTIFACT_ROOT, run=RUN_VISUALIZE)

quant_dir = ARTIFACT_ROOT / "visualizations" / "quantitative"
show_image(quant_dir / "09_summary_table.png", width=1000)
show_image(quant_dir / "01_metrics_grouped_bar.png", width=1000)
show_image(quant_dir / "02_delta_vs_baseline.png", width=900)
show_image(quant_dir / "06_validation_map_rank1_curves.png", width=1000)
show_image(quant_dir / "03_accuracy_latency_tradeoff.png", width=900)

## 11. Top-K retrieval comparison giữa 4 mô hình

Trong Re-ID, model không trả về một nhãn duy nhất mà trả về **danh sách gallery đã được xếp hạng**.

Vì vậy khi xem Top-K, ta tập trung vào:

- Ảnh đúng identity xuất hiện ở vị trí nào.
- Mức độ ổn định ranking giữa các model.
- Những trường hợp nhầm lẫn do người mặc đồ tương tự hoặc bối cảnh khó.

In [ ]:
cmd = [
    PYTHON, str(TOOLS_DIR / "visualize_multimodel_retrieval.py"),
    "--dataset-root", str(DATASET_ROOT),
    "--output-dir", str(ARTIFACT_ROOT / "visualizations" / "qualitative"),
    "--topk", "5",
    "--num-queries", "3",
    "--num-distractors", "60",
    "--batch-size", "32",
]
run_command(cmd, cwd=ARTIFACT_ROOT, run=RUN_VISUALIZE)

qual_dir = ARTIFACT_ROOT / "visualizations" / "qualitative"
show_image(qual_dir / "01_same_query_topk_comparison.png", width=950)
show_image(qual_dir / "02_true_match_rank_matrix.png", width=1000)

## 12. Local Reliability visualization

Ý tưởng reliability: không phải patch nào cũng đáng tin như nhau trong Re-ID.

- Vùng rõ ràng (thân người, texture quần áo) thường có độ tin cậy cao hơn.
- Vùng dễ nhiễu (nền, che khuất, motion blur) nên đóng góp ít hơn.

Heatmap reliability giúp nhìn trực quan mô hình đang ưu tiên vùng nào khi tạo embedding.

In [ ]:
local_out = ARTIFACT_ROOT / "visualizations" / "local_reliability"
cmd = [
    PYTHON, str(LOCAL_SRC / "visualize_person_reid_suite.py"),
    "--checkpoint", str(BRANCHES["local"]["checkpoint"]),
    "--config-file", str(BRANCHES["local"]["config"]),
    "--dataset-root", str(DATASET_ROOT),
    "--output-dir", str(local_out),
    "--num-queries", "3",
    "--topk", "5",
    "--batch-size", "16",
]
run_command(cmd, cwd=LOCAL_SRC, run=RUN_VISUALIZE)

show_image(local_out / "01_ranked_topk_results.png", width=1000)
show_image(local_out / "03_multi_query_retrieval_grid.png", width=1000)
show_image(local_out / "05_failure_cases.png", width=850)
show_image(local_out / "07_reliability_heatmap.png", width=900)
show_image(local_out / "08_synthetic_occlusion_target.png", width=950)
show_image(local_out / "06_embedding_pca.png", width=800)

## 13. Semantic Alignment visualization

Ý tưởng semantic alignment: gắn thông tin body-part/semantic mask cho patch token để embedding có cấu trúc hơn.

- Phân biệt patch thuộc vùng đầu/thân/chân.
- Giảm nhiễu từ background patch.
- Tăng tính nhất quán semantic của cùng identity qua nhiều camera.

Một số script human parser có thể cần model cache hoặc internet nếu chạy lại từ đầu.

In [ ]:
sem_out = ARTIFACT_ROOT / "visualizations" / "semantic"

cmd_basic = [
    PYTHON, str(SEMANTIC_SRC / "visualize_semantic_no_checkpoint.py"),
    "--dataset-root", str(DATASET_ROOT),
    "--output-dir", str(sem_out),
    "--num-samples", "3",
]
run_command(cmd_basic, cwd=SEMANTIC_SRC, run=RUN_VISUALIZE)

# Hai script dưới đây dùng human parser thật. Nếu máy chưa cache model, lần đầu có thể cần tải model.
cmd_real = [
    PYTHON, str(SEMANTIC_SRC / "visualize_real_semantic_examples.py"),
    "--dataset-root", str(DATASET_ROOT),
    "--output-dir", str(sem_out),
    "--num-images", "2",
]
cmd_extra = [
    PYTHON, str(SEMANTIC_SRC / "visualize_semantic_extra.py"),
    "--dataset-root", str(DATASET_ROOT),
    "--output-dir", str(sem_out),
]

run_command(cmd_real, cwd=SEMANTIC_SRC, run=RUN_VISUALIZE)
run_command(cmd_extra, cwd=SEMANTIC_SRC, run=RUN_VISUALIZE)

show_image(sem_out / "01_real_semantic_examples.png", width=1000)
show_image(sem_out / "02_real_vs_heuristic_mask.png", width=1000)
show_image(sem_out / "03_local_group_patch_targets.png", width=900)
show_image(sem_out / "04_semantic_patch_distribution.png", width=1000)
show_image(sem_out / "05_same_identity_semantic_consistency.png", width=980)
show_image(sem_out / "06_semantic_pipeline_diagram.png", width=1000)
show_image(sem_out / "07_semantic_overlay_grid.png", width=1000)

## 14. Failure case analysis

Đây là phần quan trọng khi thuyết trình: không chỉ nêu kết quả tốt mà còn chỉ ra các tình huống model nhầm.

Khi xem failure cases, nên phân tích theo 3 nhóm:

1. Occlusion hoặc crop thiếu thông tin nhận diện.
2. Similar clothing giữa nhiều người.
3. Góc nhìn/camera quá khác làm embedding lệch.

Những quan sát này giúp đề xuất hướng cải tiến tiếp theo thay vì kết luận quá mức.

In [ ]:
failure_candidates = [
    ARTIFACT_ROOT / "visualizations" / "local_reliability" / "05_failure_cases.png",
    ARTIFACT_ROOT / "visualizations" / "qualitative" / "01_same_query_topk_comparison.png",
    ARTIFACT_ROOT / "visualizations" / "qualitative" / "02_true_match_rank_matrix.png",
]

shown = 0
for img in failure_candidates:
    if img.exists():
        show_image(img, width=1100)
        shown += 1

if shown == 0:
    print("Chưa có artifact failure/qualitative. Hãy bật RUN_VISUALIZE = True để tạo lại hình.")

## 15. Optional: Train command reference

Phần này chỉ để tham khảo lệnh train. **Không nên chạy train khi quay video demo** vì rất tốn thời gian.

Khi quay video, ưu tiên dùng checkpoint đã có và tập trung vào evaluation + visualization.

In [ ]:
def build_train_command(key, fast_demo=True):
    branch = BRANCHES[key]
    output_dir = DEMO_OUTPUT / f"train_{key}"
    cmd = [
        PYTHON, "train.py",
        "--config_file", branch["config"],
        "DATASETS.ROOT_DIR", str(DATA_PARENT),
        "MODEL.DEVICE_ID", DEVICE_ID,
        "OUTPUT_DIR", str(output_dir),
    ]
    if fast_demo:
        cmd += [
            "MODEL.PRETRAIN_CHOICE", "none",
            "SOLVER.MAX_EPOCHS", "1",
            "SOLVER.EVAL_PERIOD", "1",
            "SOLVER.CHECKPOINT_PERIOD", "1",
            "SOLVER.IMS_PER_BATCH", "8",
            "TEST.IMS_PER_BATCH", "64",
            "DATALOADER.NUM_WORKERS", "0",
        ]
    return cmd

for key in ["baseline", "local", "semantic", "combine"]:
    display(Markdown(f"### Train reference: {BRANCHES[key]['title']}"))
    cmd = build_train_command(key, fast_demo=FAST_TRAIN_DEMO)
    run_command(cmd, cwd=BRANCHES[key]["source"], run=RUN_TRAIN)

## 16. Script gợi ý khi quay video

Thứ tự trình bày ngắn gọn (tiếng Việt):

1. **Giới thiệu bài toán**: Person Re-ID là bài toán truy hồi theo ranking giữa query và gallery, không phải classification 1 nhãn.
2. **Giới thiệu baseline + 3 cải tiến**: Baseline TransReID, Local Reliability, Semantic Alignment, và Semantic + Reliability.
3. **Kiểm tra dữ liệu/checkpoint**: chạy preflight để xác nhận dataset, config, checkpoint, script đều sẵn sàng.
4. **Chạy evaluation từ checkpoint**: bật `RUN_EVAL = True` để lấy metric của 4 mô hình.
5. **Phân tích metric**: nhấn vào mAP, Rank-1/5/10 và delta so với baseline.
6. **Phân tích Top-K retrieval**: so sánh vị trí true match giữa các mô hình cho cùng query.
7. **Phân tích reliability/semantic**: trình bày heatmap reliability và semantic alignment để giải thích vì sao model cải thiện.
8. **Kết luận + hạn chế**: có cải thiện xu hướng trên Market-1501, nhưng vẫn còn failure cases và cần đánh giá thêm trên dữ liệu khó hơn.